In [ ]:
1. read pickled matrix, map ensp to uniport, include merge and delete
2. cv select beta and merge

In [1]:
import pickle
import pandas as pd
import numpy as np
import networkx as nx
import mygene

In [2]:
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019.txt'
G = nx.read_edgelist(file_path, nodetype=str, create_using=nx.Graph())
ensps = list(G.nodes())
del G

In [4]:
def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df

In [9]:
ppi_ids_map = get_map_df([s.split('.')[1] for s in ensps],'ensembl.protein')

ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = ppi_ids_map[ppi_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('ENSP00000473163', 3)]
1939 input query terms found no hit:	['ENSP00000415070', 'ENSP00000267197', 'ENSP00000415452', 'ENSP00000006101', 'ENSP00000262477', 'ENS


In [ ]:
more2one_df['string_id'] = '9606.'+more2one_df['query']
merge_dict = more2one_df.groupby('uniprot_ids')['string_id'].apply(list).to_dict()
map_dict = dict()
merge_groups = []
for key in merge_dict:
    merge_groups.append(merge_dict[key])
    new_key = '_'.join(sorted(merge_dict[key]))
    map_dict[new_key] = key
flat_set = {item for sublist in merge_groups for item in sublist}
unique_ids = ['9606.' + ensp_id for ensp_id in ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]
delete_list = list(set(ensps) - flat_set - set(unique_ids))
sample_names = ensps

In [31]:
len(flat_set),len(unique_ids),len(ensps),len(delete_list)

(168, 17151, 19576, 2257)

In [ ]:

with open('/itf-fi-ml/shared/users/ziyuzh/svm/results/df/2019/difussion_K_0.1.pkl', 'rb') as f:
    sim_matrix = pickle.load(f)

In [33]:
# Step 1: Convert to tidy format
df = pd.DataFrame(sim_matrix, index=sample_names, columns=sample_names)
tidy = df.stack().reset_index()
tidy.columns = ['Sample_i', 'Sample_j', 'Similarity']
tidy = tidy[tidy['Sample_i'] < tidy['Sample_j']]  # remove duplicates

# Step 2: Precompute all pair similarities
# For quick lookup, use frozen set of pair as key
pair_sim = {
    frozenset([i, j]): s for i, j, s in tidy.itertuples(index=False)
}

# Step 3: Build new groups
sample_to_group = {}
new_names = []
for idx, group in enumerate(merge_groups):
    new_name = '_'.join(sorted(group))
    new_names.append(new_name)
    for s in group:
        sample_to_group[s] = new_name

# Samples not in any merge group or deletion
all_samples = set(sample_names)
merged_samples = set(sample_to_group.keys())
kept_samples = sorted(all_samples - merged_samples - set(delete_list))

# Step 4: Build final sample list and group mapping
final_samples = new_names + kept_samples
group_map = {name: [name] for name in kept_samples}
for group in merge_groups:
    group_name = '_'.join(sorted(group))
    group_map[group_name] = group

# Step 5: Compute average similarity between groups
records = []
for i, group_i in enumerate(final_samples):
    for j in range(i, len(final_samples)):
        group_j = final_samples[j]
        members_i = group_map[group_i]
        members_j = group_map[group_j]
        
        # Compute all pairwise similarities
        sims = []
        for a in members_i:
            for b in members_j:
                if a != b:
                    key = frozenset([a, b])
                    if key in pair_sim:
                        sims.append(pair_sim[key])
        # Self-similarity
        sim_val = 1.0 if group_i == group_j else np.mean(sims)
        records.append((group_i, group_j, sim_val))
        if group_i != group_j:
            records.append((group_j, group_i, sim_val))

# Step 6: Build final tidy and square matrix
new_tidy = pd.DataFrame(records, columns=['Sample_i', 'Sample_j', 'Similarity'])
new_matrix = new_tidy.pivot(index='Sample_i', columns='Sample_j', values='Similarity')
new_matrix = new_matrix.reindex(index=final_samples, columns=final_samples, fill_value=1.0)

: 

In [ ]:
new_matrix.rename(index=map_dict, columns=map_dict, inplace=True)

In [ ]:
def process_kernel(args):
    K= args
    eigenvalues, eigenvectors = np.linalg.eigh(K)
    eigenvalues = np.clip(eigenvalues, 1e-12, None)  # Avoid log(0)
    K_log = eigenvectors @ np.diag(np.log(eigenvalues)) @ eigenvectors.T
    K_log = 0.5 * (K_log + K_log.T)

    return K_log
def normalize_kernel(K):
    diag = np.sqrt(np.diag(K))
    diag[diag == 0] = 1e-8  # Avoid division by zero
    return K / (diag[:, None] * diag[None, :])

In [ ]:
K_full = new_matrix
K_full = normalize_kernel(K_full)
K_full += np.eye(K_full.shape[0]) * 1e-6
K_full = 0.5 * (K_full + K_full.T)
logm_k = process_kernel(K_full)